## 1 — ライブラリの読込み

- 処理: ライブラリの読込み
- 入力: プロジェクトのPython環境
- 出力: 結果読込み・描画・明示操作の関数
- 次: 実験JSONを指定

- 表はクリックで展開 / 縦横スクロール / CSVは同じ列順の全行・全精度

- 共通評価は新規発見・継続的な嗜好・多様性／発見性・効率性 / 集計は02、定義は[評価仕様](../docs/listening-evaluation.md)


In [ ]:
import json
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from music_recsys.evaluation_views import show_accuracy, show_other_aspects
from music_recsys.experiment_search import (
    freeze_selection,
    load_study_config,
    ranked_settings,
    select_best,
)
from music_recsys.full_analysis import curves as full_curves
from music_recsys.full_analysis import is_full
from music_recsys.notebook_config import ExperimentConfigPicker
from music_recsys.notebook_display import (
    show_state,
    show_table,
)
from music_recsys.search_analysis import (
    final_comparison,
    history_performance,
    parameter_sensitivity,
    plot_search,
    plot_sensitivity,
    search_table,
    selected_curves,
    trial_curves,
    workflow_state,
)

## 2 — 実験設定を選択

- 処理: configs内のlistening-v2設定を選択し、読み込むボタンで確定
- 入力: 選択欄のJSON / 初期状態は未選択
- 出力: 読み込んだ実験名・モデル・入力パス / 評価は開始しない
- 次: 読込み済みの表示を確認して次セルへ / ファイル追加後は一覧を更新


In [ ]:
ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
config_picker = ExperimentConfigPicker(ROOT / "configs")
display(config_picker.widget)

## 3 — 選択した設定の確認

- 処理: 明示的に読み込んだ設定を後続セルで使用
- 入力: 選択欄の読込み済み設定
- 出力: WORKFLOWと実験名 / 未指定なら案内して停止
- 次: 後続セルを上から順に実行 / 別のNotebookでも同じファイルを選択


In [ ]:
WORKFLOW = config_picker.require_loaded()
EXPERIMENT_CONFIG = WORKFLOW.path
print("実験:", WORKFLOW.name)

## 4 — 設定と実行状態の確認

- 処理: 設定と実行状態の確認
- 入力: WORKFLOWと保存済みstudy
- 出力: モデル・主条件・探索設定・現在の状態
- 次: Validation試行の一覧へ


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
workflow = workflow_state(WORKFLOW)
settings = {
    "モデル": WORKFLOW.model,
    "主タスク": WORKFLOW.task or "未指定",
    "主指標": WORKFLOW.metric or "未指定",
    "評価期間（日）": WORKFLOW.evaluation.primary_window_days,
    "日数重み": WORKFLOW.evaluation.primary_day_weight,
    "推薦件数 K": WORKFLOW.primary_k,
    "seed": ", ".join(map(str, WORKFLOW.seeds)),
    "探索する設定値": str(WORKFLOW.grid) if WORKFLOW.grid else "なし",
    "最大試行数": WORKFLOW.max_trials,
}
show_state(workflow)
show_table(
    pd.Series(settings, name="値").rename_axis("項目").reset_index(), "実験条件", "02-settings"
)

## 5 — Validation試行の一覧

- 処理: 成功試行と失敗試行を分けて表示
- 入力: 保存済みValidation試行 / 指標は設定した主条件・Kで集計
- 出力: 成功表は試行ID・seed・モデル設定値・主指標・試行時間 / 失敗表はエラー種別
- parametersはモデルの設定値 / Popularityには設定値がないため非表示
- 試行時間は処理全体の秒数 / 推薦1件のlatencyではない
- 失敗原因の詳細は現在未保存 / エラー種別だけで原因を断定しない
- 次: 成功試行の性能を確認 / 失敗があればログを確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
workflow = workflow_state(WORKFLOW)
search_df = pd.DataFrame()
if not is_full(WORKFLOW) and workflow["status"] != "mismatch":
    search_df = search_table(WORKFLOW)
if is_full(WORKFLOW):
    print("全件Popularity / 結果は次のセルで表示")
elif search_df.empty:
    print("探索結果は未取得 / 01の確認後、00でSmoke → searchを実行してください")
else:
    successful = search_df[search_df.status == "succeeded"]
    failed = search_df[search_df.status == "failed"]
    pending = search_df[~search_df.status.isin(["succeeded", "failed"])]
    print(f"成功 {len(successful)}件 / 失敗 {len(failed)}件 / その他 {len(pending)}件")
    search_fig = plot_search(WORKFLOW)
    if search_fig is not None:
        plt.show()
    elif len(WORKFLOW.grid) > 2:
        sensitivity_df = parameter_sensitivity(WORKFLOW)
        show_table(sensitivity_df, "パラメータ別の値", "02-sensitivity")
        for parameter in sensitivity_df.varied_parameter.unique():
            plot_sensitivity(
                sensitivity_df[sensitivity_df.varied_parameter == parameter],
                metric=WORKFLOW.metric or "ndcg (diagnostic)",
                k=WORKFLOW.primary_k,
            )
            plt.show()
        print("他の条件はfixed_parametersに固定")
    elif WORKFLOW.grid:
        print("描画できる探索変数・成功値がありません")
    if not successful.empty:
        parameter_columns = [c for c in search_df if c.startswith("param:")]
        metric_columns = [WORKFLOW.metric] if WORKFLOW.metric else []
        columns = [*parameter_columns, "seed", *metric_columns, "seconds", "trial_id"]
        show_table(
            successful[[c for c in columns if c in successful]]
            .rename(columns={c: c[6:] for c in parameter_columns})
            .reset_index(drop=True),
            "成功試行",
            "02-successful-trials",
        )
        if WORKFLOW.metric is None:
            print("主指標は未指定 / 指標別の診断値は次の性能表で確認")
    if not failed.empty:
        print("失敗した試行")
        show_table(
            failed[["seed", "error_type", "trial_id"]]
            .rename(columns={"error_type": "エラー種別"})
            .fillna("未記録")
            .assign(原因詳細="未保存 / 実行ログを確認")
            .reset_index(drop=True),
            "失敗試行",
            "02-failed-trials",
        )
        print("ログ保存先:", WORKFLOW.output / "run.log")
    if not pending.empty:
        print("未完了などの試行")
        show_table(
            pending[["status", "seed", "trial_id"]].reset_index(drop=True),
            "未完了試行",
            "02-pending-trials",
        )
print("次:", workflow["next_action"])

## 6 — 新規発見・継続的な嗜好

- 処理: 全モデル共通の指標をK=2〜20で描画
- 入力: 選択設定 / 未選択なら指定した成功試行
- 出力: 新規発見はBinary NDCG・Recall / 継続的な嗜好はGraded NDCG・Repeat Recall
- Graded NDCGは未聴曲の定着代理と既聴曲を分ける / 期間内の観測であり継続嗜好の確定ではない
- 帯はユーザー単位95% CI / 再生回数・再生日数の分布は01で確認
- 次: 多様性・発見性・効率性へ


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
DIAGNOSTIC_TRIAL_ID = None  # 指定する場合は上の探索表のtrial_id
BOOTSTRAP_RESAMPLES = 2000
workflow_curves = pd.DataFrame()
workflow = workflow_state(WORKFLOW)
if is_full(WORKFLOW) and workflow["status"] == "validation_complete":
    workflow_curves = full_curves(WORKFLOW)
elif (WORKFLOW.output / "selection.json").exists() and workflow["status"] != "mismatch":
    workflow_curves = selected_curves(WORKFLOW, resamples=BOOTSTRAP_RESAMPLES)
elif not search_df.empty and (search_df.status == "succeeded").any():
    successful = search_df[search_df.status == "succeeded"]
    diagnostic_id = DIAGNOSTIC_TRIAL_ID or successful.iloc[0].trial_id
    print("診断表示 / 最良設定の選択ではありません / trial_id:", diagnostic_id)
    workflow_curves = trial_curves(WORKFLOW, diagnostic_id, resamples=BOOTSTRAP_RESAMPLES)
if workflow_curves.empty:
    print("表示できる成功試行がありません / 00で実行状態を確認してください")
else:
    show_accuracy(workflow_curves)

## 7 — 多様性・発見性・効率性

- 処理: 全モデル共通の残り2観点を表示
- 入力: 同じ保存済みValidation結果 / K=2〜20
- 出力: ILD・Catalog Coverage・NoveltyのK曲線 / 平均レイテンシ・入出力トークン・API料金・LLM call・tool call
- 効率性は1回の生成に対して集計 / 未測定は欠測として表示
- 多様性の評価人数・除外人数も表に記載 / 選択済みなら履歴量別性能を表示
- 次: 選択用の主指標を事前指定した実験なら設定選択へ


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
beyond_df = pd.DataFrame()
efficiency_df = pd.DataFrame()
if not workflow_curves.empty:
    beyond_df, efficiency_df = show_other_aspects(workflow_curves)
    if (WORKFLOW.output / "selection.json").exists():
        show_table(history_performance(WORKFLOW), "履歴量別の性能", "02-history")
else:
    print("成功試行の結果を確認してから実行してください")

## 8 — 設定の選択

- 処理: 設定の選択
- 入力: SELECT = False / 明示時のみTrue・事前指定した主指標
- 出力: 順位表 / Trueならselection.jsonを保存
- 次: 03_recommendationで推薦を確認


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
SELECT = False
if SELECT:
    selection = select_best(WORKFLOW)
    print("選択を保存しました")
workflow = workflow_state(WORKFLOW)
if WORKFLOW.metric and workflow["status"] in {
    "selection_pending",
    "review_pending",
    "review_rejected",
    "freeze_pending",
    "test_ready",
    "test_succeeded",
    "test_failed",
}:
    ranking = pd.DataFrame(ranked_settings(WORKFLOW))
    columns = ["score", "seed_std", "mean_seconds", "seeds"]
    if any(ranking.parameters):
        columns.insert(0, "parameters")
    show_table(ranking[columns], "主指標による設定順位", "02-ranking")
print("次:", workflow["next_action"])

## 9 — 確認済み設定の固定

- 処理: 確認済み設定の固定
- 入力: FREEZE = False / 明示時のみTrue・03でのaccept記録
- 出力: Trueなら固定artifactを保存
- 次: 固定後はCLI test / 未確認なら03へ


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
FREEZE = False
if FREEZE:
    locked = freeze_selection(WORKFLOW)
    print("設定を固定しました")
show_state(workflow_state(WORKFLOW))

## 10 — 固定後のTest比較

- 処理: 固定後のTest比較
- 入力: 同条件の固定済みTest結果・comparison.configs
- 出力: モデル別状態・最終比較 / このセルはTestを実行しない
- 次: 結果を記録 / Testを使って再探索しない


In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
experiment_settings = (
    json.loads(EXPERIMENT_CONFIG.read_text())
    if EXPERIMENT_CONFIG.suffix == ".json"
    else tomllib.loads(EXPERIMENT_CONFIG.read_text())
)
comparison_paths = [WORKFLOW.path] + [
    (WORKFLOW.path.parent / p).resolve()
    for p in experiment_settings.get("comparison", {}).get("configs", [])
]
comparison_configs = [load_study_config(p) for p in dict.fromkeys(comparison_paths)]
show_table(
    pd.DataFrame(
        [{"experiment": c.name, "状態": workflow_state(c)["label"]} for c in comparison_configs]
    ),
    "Test実行状態",
    "02-test-status",
)
final_df = pd.DataFrame()
try:
    final_df = final_comparison(comparison_configs) if not is_full(WORKFLOW) else pd.DataFrame()
    if final_df.empty:
        print("Testは未実行 / 03で確認 → 設定固定 → CLI testの順に進めてください")
    else:
        show_accuracy(final_df, "test")
        show_other_aspects(final_df, "test")
        print("Testの比較条件を統一 / 探索回数・総時間・seed数も併せて報告してください")
except ValueError as error:
    print("最終比較を停止:", error)

## 評価期間とK
- 処理: 指標ごとのヒートマップを表示
- 入力: 保存済み全件結果
- 出力: K × 評価期間の図
- 次: 正確な値はmetrics.csvで確認

In [ ]:
config_picker.require_current(globals().get("WORKFLOW"))
if is_full(WORKFLOW) and not workflow_curves.empty:
    from music_recsys.evaluation_figures import accuracy_heatmaps, show_figures

    show_figures(accuracy_heatmaps(workflow_curves, WORKFLOW.evaluation.primary_day_weight))